In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, make_scorer, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Using robust ensemble methods for the imbalanced cluster 2
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

In [2]:
# Load the cluster-specific data
df_cluster2 = pd.read_csv("cluster_2.csv")
print("Data loaded. Shape:", df_cluster2.shape)

Data loaded. Shape: (2059, 98)


In [3]:
# Load the features used for clustering (Task A's output)
top40 = joblib.load("top_features_for_clustering.joblib")
features_to_use = top40

X_sub = df_cluster2[features_to_use].copy()
y_sub = df_cluster2["Bankrupt?"].copy()

print("Cluster 2 shape:", X_sub.shape)
print("Target Distribution:\n", y_sub.value_counts())

Cluster 2 shape: (2059, 40)
Target Distribution:
 Bankrupt?
0    2053
1       6
Name: count, dtype: int64


In [4]:
rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight={0: 1, 1: 342},  # explicit weight
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lgbm = LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.01,
    num_leaves=31,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.01,
    scale_pos_weight=342,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='logloss'
)

base_estimators = [
    ('rf', rf),
    ('lgbm', lgbm),
    ('xgb', xgb),
]

In [5]:
# Defining Meta Model
meta_model = LogisticRegression(
    penalty="l2",
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Stacking Pipeline
# Note: Using StratifiedKFold with cv=3 because Cluster 2 has only 6 positives.
# cv=5 would risk folds having 0 or 1 positive sample, making training unstable.
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1
)

# Pipeline: Scale the data first -> Stacking
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('stacking', stacking_clf)
])

In [6]:
# Model fitting
print("Fitting model...")
model_pipeline.fit(X_sub, y_sub)

# Predict on the original cluster-2 train rows
y_pred = model_pipeline.predict(X_sub)

# Compute Confusion Matrix and Eq(1) Accuracy
cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# TT: True Bankrupts Caught (True Positives)
TT = int(tp)
# TF: Bankrupts Missed (False Negatives)
TF = int(fn)
N_features = len(features_to_use)

# Eq (1) Accuracy: TT / (TF + TT) -> Recall for the positive class
eq1_acc = TT / (TF + TT) if (TF + TT) > 0 else 0

print("-" * 40)
print("RESULTS FOR TABLE 3 (Cluster 2 Model C)")
print("-" * 40)
print(f"Confusion Matrix:\n{cm}")
print(f"TT (True Bankrupts Caught): {TT}")
print(f"TF (Bankrupts Missed):      {TF}")
print(f"N_features:                 {N_features}")
print(f"Eq(1) Accuracy:             {eq1_acc:.4f}")
print("-" * 40)

Fitting model...
----------------------------------------
RESULTS FOR TABLE 3 (Cluster 2 Model C)
----------------------------------------
Confusion Matrix:
[[2024   29]
 [   0    6]]
TT (True Bankrupts Caught): 6
TF (Bankrupts Missed):      0
N_features:                 40
Eq(1) Accuracy:             1.0000
----------------------------------------


In [7]:
cluster2_package = {
    "cluster_id": 2,
    "features": features_to_use,
    "pipeline": model_pipeline, # Contains both scaler and stacking model
    "table3_stats": {"TT": TT, "TF": TF, "Eq1_acc": eq1_acc, "N_features": N_features}
}

joblib.dump(cluster2_package, "cluster2_stacking_C.joblib")
print("Saved cluster2_stacking_C.joblib")

Saved cluster2_stacking_C.joblib
